In [1]:
import pandas as pd

# Load the dataset from the project's data folder
data = pd.read_csv("../data/sample_data.csv")

# Display the first few rows
data.head()

,id,question,text_type,label,text
0,1_human,what composer used sound mass,human,0,"Composers and works include Barbara Kolb , Pau..."
1,1_ai_original,what composer used sound mass,ai_original,1,Sound mass is a term used to describe a type o...
2,2_human,where did the persian war take place,human,0,The Greco-Persian Wars (also often called the ...
3,2_ai_original,where did the persian war take place,ai_original,1,The Persian Wars were a series of conflicts fo...
4,3_human,what are add ons,human,0,"Plug-in (computing) , a piece of software whic..."


In [2]:
# Check the number of rows and columns
print("Dataset shape:", data.shape)

# Check how many passages we have of each type
print("\nText type counts:")
print(data["text_type"].value_counts())

# Check the label counts
print("\nLabel counts:")
print(data["label"].value_counts())

Dataset shape: (1000, 5)

Text type counts:
text_type
human          500
ai_original    500
Name: count, dtype: int64

Label counts:
label
0    500
1    500
Name: count, dtype: int64


In [9]:
from sklearn.model_selection import train_test_split

# Separate the text and labels
X = data["text"]
y = data["label"]

# Use 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining label distribution:")
print(y_train.value_counts())

print("\nTesting label distribution:")
print(y_test.value_counts())

Training samples: 800
Testing samples: 200

Training label distribution:
label
1    400
0    400
Name: count, dtype: int64

Testing label distribution:
label
1    100
0    100
Name: count, dtype: int64


In [10]:
import sys

# Add the project root folder to Python's module search path
sys.path.append("..")

In [11]:
from src.preprocessing import create_tfidf_features

# Convert the training and testing text into TF-IDF features
X_train_tfidf, X_test_tfidf, vectorizer = create_tfidf_features(
    X_train,
    X_test
)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (800, 9024)
Testing TF-IDF shape: (200, 9024)


In [12]:
from src.model import create_model, train_model, predict

# Create the Logistic Regression model
model = create_model()

# Train the model using the TF-IDF training data
model = train_model(model, X_train_tfidf, y_train)

print("Model trained successfully.")

Model trained successfully.


In [13]:
# Make predictions on the unseen test data
y_pred = predict(model, X_test_tfidf)

print("Predictions:", y_pred)


Predictions: [1 0 0 1 1 1 0 0 1 1 0 1 1 0 1 0 0 1 1 0 1 0 0 1 0 0 0 0 0 1 0 1 1 1 1 0 0
 0 0 1 1 1 1 1 0 0 0 1 1 0 1 0 0 0 0 0 1 1 0 1 1 1 0 1 1 1 0 1 0 0 0 1 1 0
 1 1 0 0 1 0 1 1 1 0 1 1 1 1 0 0 1 1 0 0 1 1 1 0 0 0 0 1 0 1 1 0 0 1 0 1 1
 1 0 0 0 1 1 0 0 0 1 1 0 1 1 0 0 1 0 0 0 1 1 0 1 0 1 0 0 0 1 0 1 0 1 1 0 0
 1 0 0 0 0 1 1 0 1 0 1 0 1 0 1 0 0 1 1 0 1 1 0 1 1 0 1 1 0 1 1 0 1 1 1 1 1
 1 1 0 1 1 0 1 0 1 0 1 1 1 1 1]


In [14]:
from src.evaluation import evaluate_model

# Compare the model's predictions with the actual labels
results = evaluate_model(y_test, y_pred)

# Display each evaluation metric
for metric, value in results.items():
    print(f"{metric}: {value:.4f}")

accuracy: 0.6550
precision: 0.6449
recall: 0.6900
f1_score: 0.6667
false_positive_rate: 0.3800


In [15]:
from sklearn.metrics import confusion_matrix

# Show how the model's predictions compare with the actual labels
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[62 38]
 [31 69]]
